In [1]:
import pandas as pd

df = pd.read_csv("./data_cleaned.csv").fillna("")

def merge_rows(group):
    texts = []

    for _, row in group.iterrows():
        if row["heading"]:
            texts.append(f"## {row['heading']}")
        if row["content"]:
            texts.append(row["content"])

    return pd.Series({
        "title": group.iloc[0]["title"],
        "url": group.iloc[0]["url"],
        "text": "\n\n".join(texts)
    })

documents = (
    df.groupby("url", as_index=False)
      .apply(merge_rows)
      .reset_index(drop=True)
)

print(documents.head())

                                               title  \
0   40 tuổi, bị bệnh lupus muốn có con phải làm sao?   
1         AMH cao có thể có con tự nhiên được không?   
2  AMH là 0,1 có làm IVF bằng trứng của mình được...   
3  Bị buồng trứng đa nang, chưa có trứng trội thì...   
4  Bị nhiễm sắc thể số 9 thì có mang thai được kh...   

                                                 url  \
0  https://tamanhhospital.vn/tu-van/40-tuoi-bi-be...   
1  https://tamanhhospital.vn/tu-van/amh-cao-co-co...   
2  https://tamanhhospital.vn/tu-van/amh-la-01-co-...   
3  https://tamanhhospital.vn/tu-van/bi-buong-trun...   
4  https://tamanhhospital.vn/tu-van/bi-nhiem-sac-...   

                                                text  
0  ## 40 tuổi, bị bệnh lupus muốn có con phải làm...  
1  ## AMH cao có thể có con tự nhiên được không?\...  
2  ## AMH là 0,1 có làm IVF bằng trứng của mình đ...  
3  ## Bị buồng trứng đa nang, chưa có trứng trội ...  
4  ## Bị nhiễm sắc thể số 9 thì có mang thai được..

C:\Users\Windows\AppData\Local\Temp\ipykernel_13676\1744706897.py:22: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(merge_rows)


In [ ]:

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "123456789"

Import Document

In [ ]:
from neo4j import GraphDatabase
import json

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=(USERNAME, PASSWORD)
)

def create_document(tx, row):
    query = """
    MERGE (d:Document {url: $url})

    SET d.title = $title,
        d.text = $text,
        d.metadata = $metadata,
        d.text_unit_ids = $text_unit_ids
    """

    tx.run(
        query,
        url=row["url"],
        title=row["title"],
        text=row["text"],
        metadata=json.dumps({"url": row["url"]}),
        text_unit_ids=[]
    )

with driver.session() as session:
    for _, row in documents.iterrows():
        session.execute_write(create_document, row)

driver.close()

Improt Chunk

In [ ]:
from neo4j import GraphDatabase
import pandas as pd

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=(USERNAME, PASSWORD)
)

df_chunk = pd.read_csv("./content_chunks.csv").fillna("")


def create_chunk(tx, row):
    query = """
    // Bước 1: Tìm Document dựa trên URL có trong file CSV của Chunk
    MATCH (d:Document {url: $url})

    // Bước 2: Tạo hoặc tìm Chunk
    MERGE (c:Chunk {chunk_id: $chunk_id})

    // Bước 3: Gán thuộc tính cho Chunk
    SET
        c.chunk_id = $chunk_id,
        c.chunk_text = $chunk_text,
        c.entity_ids = $entity_ids,
        c.relationships_ids = $relationships_ids,
        c.document_id = elementId(d)

    // Bước 4: Cập nhật text_unit_ids trong node Document
    // Sử dụng biểu thức điều kiện (coalesce) để phòng trường hợp d.text_unit_ids chưa được khởi tạo (null)
    SET d.text_unit_ids = coalesce(d.text_unit_ids, []) + [$chunk_id]

    // Bước 5: Tạo mối quan hệ giữa 2 Node
    MERGE (d)-[:HAS_CHUNK]->(c)
    """

    tx.run(
        query,
        chunk_id=row["chunk_id"],
        chunk_text=row["chunk_text"],
        entity_ids=[],
        relationships_ids=[],
        url=row["url"]
    )

with driver.session() as session:
    for _, row in df_chunk.iterrows():
        session.execute_write(create_chunk, row)

driver.close()

Import Entity

In [6]:
import ast
from neo4j import GraphDatabase
import pandas as pd

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=(USERNAME, PASSWORD)
)

df = pd.read_csv("./content_chunks_extracted_1-100.csv").fillna("")

def create_entities(tx, row):
    try:
        extracted = ast.literal_eval(row["extracted_text"])
    except Exception as e:
        print(f"Cannot parse chunk {row['chunk_id']}: {e}")
        return

    chunk_id = int(row["chunk_id"])

    for idx, entity in enumerate(extracted.get("entities", [])):
        mention_id = f"{chunk_id}_{idx}"

        tx.run(
            """
            // Bước 1: Tìm Chunk đang xử lý
            MATCH (c:Chunk {chunk_id: $chunk_id})

            // Bước 2: Tạo EntityMention mới
            CREATE (e:EntityMention {
                mention_id: $mention_id,
                chunk_id: $chunk_id,
                name: $name,
                type: $type,
                description: $description,
                chunk_ids: [$chunk_id],
                frequency: 1,
                degree: 0
            })

            // Bước 3: Quay ngược lại cập nhật mảng entity_ids bên trong node Chunk
            // Nếu c.entity_ids chưa được khởi tạo (null), coalesce sẽ tự biến thành mảng rỗng [] để cộng
            SET c.entity_ids = coalesce(c.entity_ids, []) + [$mention_id]

            // Bước 4: Tạo mối quan hệ từ Chunk sang EntityMention
            CREATE (c)-[:MENTIONS]->(e)
            """,
            chunk_id=chunk_id,
            mention_id=mention_id,
            name=entity.get("entity_name", ""),
            type=entity.get("entity_type", ""),
            description=entity.get("entity_description", "")
        )

with driver.session() as session:
    for _, row in df.iterrows():
        session.execute_write(create_entities, row)

driver.close()

Import Relationship

In [7]:
import ast
from neo4j import GraphDatabase
import pandas as pd

driver = GraphDatabase.driver(
    "bolt://localhost:7687",
    auth=(USERNAME, PASSWORD)
)
df = pd.read_csv("./content_chunks_extracted_1-100.csv").fillna("")

def create_relationships(tx, row):
    try:
        extracted = ast.literal_eval(row["extracted_text"])
    except Exception as e:
        print(f"Cannot parse chunk {row['chunk_id']}: {e}")
        return

    chunk_id = int(row["chunk_id"])

    entities = extracted.get("entities", [])
    relationships = extracted.get("relationships", [])

    name_to_mention = {}

    for idx, entity in enumerate(entities):
        mention_id = f"{chunk_id}_{idx}"
        name_to_mention[entity.get("entity_name")] = mention_id

    for rel in relationships:
        source = name_to_mention.get(rel.get("source_entity"))
        target = name_to_mention.get(rel.get("target_entity"))

        if source is None or target is None:
            print(
                f"Skip relationship in chunk {chunk_id}: "
                f"{rel.get('source_entity')} -> {rel.get('target_entity')}"
            )
            continue

        relationship_id = f"{source}_{target}_{idx}" 

        tx.run(
            """
            // Bước 1: Tìm Node Chunk và 2 Node EntityMention liên quan
            MATCH (c:Chunk {chunk_id: $chunk_id})
            MATCH (s:EntityMention {mention_id: $source})
            MATCH (t:EntityMention {mention_id: $target})

            // Bước 2: Tạo mối quan hệ giữa Source và Target
            CREATE (s)-[r:RELATED_TO {
                relationship_id: $relationship_id, // Nên lưu lại ID này trên cạnh để sau này dễ tra cứu
                source: $source,
                target: $target,
                description: $description,
                weight: $weight,
                combined_degree: 0,
                chunk_ids: [$chunk_id]
            }]->(t)

            // Bước 3: Cập nhật danh sách relationships_ids trong node Chunk
            SET c.relationships_ids = coalesce(c.relationships_ids, []) + [$relationship_id]
            """,
            chunk_id=chunk_id,
            relationship_id=relationship_id,
            source=source,
            target=target,
            description=rel.get("relationship_description", ""),
            weight=float(rel.get("relationship_strength", 0))
        )

with driver.session() as session:
    for _, row in df.iterrows():
        session.execute_write(create_relationships, row)

driver.close()

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@

Entity resolution

In [9]:

URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "123456789"

In [10]:
from neo4j import GraphDatabase
import pandas as pd

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

query = """
MATCH (e:EntityMention)
RETURN
e.mention_id AS mention_id,
e.name AS name,
e.type AS type
"""

with driver.session() as session:
    result = session.run(query)
    df = pd.DataFrame([r.data() for r in result])

driver.close()

print(df.head())


  mention_id                       name       type
0       12_0                   Pentinox       drug
1       12_1  Bệnh viện Đa khoa Tâm Anh   hospital
2       12_2                   Bệnh tật  condition
3       11_0  Bệnh viện Đa khoa Tâm Anh   hospital
4       11_1  hở khuyết sẹo mổ lấy thai  condition


In [11]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("bkai-foundation-models/vietnamese-bi-encoder")

texts = (
    df["name"].fillna("")
    + "\n"
    + df["type"].fillna("")
).tolist()


embeddings = model.encode(
    texts,
    batch_size=128,
    normalize_embeddings=True,
    show_progress_bar=True
)

Batches: 100%|██████████| 3/3 [00:00<00:00,  3.23it/s]


In [12]:
def write_embedding(tx, mention_id, embedding):
    tx.run(
        """
        MATCH (e:EntityMention {mention_id:$id})
        SET e.embedding=$embedding
        """,
        id=mention_id,
        embedding=embedding.tolist()
    )

In [13]:
BATCH_SIZE = 500

driver = GraphDatabase.driver(
    URI,
    auth=(USERNAME, PASSWORD)
)

with driver.session() as session:

    for start in range(0, len(df), BATCH_SIZE):

        end = min(start + BATCH_SIZE, len(df))

        batch = df.iloc[start:end]

        with session.begin_transaction() as tx:

            for i, row in batch.iterrows():

                write_embedding(
                    tx,
                    row["mention_id"],
                    embeddings[i]
                )

            tx.commit()

        print(end)

driver.close()


356


Group

In [14]:
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "123456789"


In [15]:
from graphdatascience import GraphDataScience

gds = GraphDataScience(
    URI,
    auth=(USERNAME, PASSWORD)
)
if gds.graph.exists("entity_mentions")["exists"]:
    gds.graph.drop("entity_mentions")

In [16]:
G, result = gds.graph.project(
    "entity_mentions",
    "EntityMention",
    "*",
    nodeProperties=["embedding"]
)

print(result)

nodeProjection            {'EntityMention': {'properties': {'embedding':...
relationshipProjection    {'__ALL__': {'orientation': 'NATURAL', 'aggreg...
graphName                                                   entity_mentions
nodeCount                                                               356
relationshipCount                                                       291
projectMillis                                                            39
Name: 0, dtype: object


In [17]:
gds.knn.mutate(
    G,
    nodeProperties=["embedding"],
    mutateRelationshipType="SIMILAR",
    mutateProperty="score",
    similarityCutoff=0.80
)

preProcessingMillis                                                       0
computeMillis                                                           147
mutateMillis                                                             98
postProcessingMillis                                                      0
nodesCompared                                                           356
relationshipsWritten                                                   1257
similarityDistribution    {'p1': 0.8003387451171875, 'p5': 0.80900192260...
didConverge                                                            True
ranIterations                                                             7
nodePairsConsidered                                                  132167
configuration             {'topK': 10, 'maxIterations': 100, 'randomJoin...
Name: 0, dtype: object

In [18]:
gds.wcc.write(
    G,
    relationshipTypes=["SIMILAR"],
    writeProperty="wcc"
)

componentCount                                                         144
componentDistribution    {'p1': 1, 'max': 57, 'p5': 1, 'p90': 3, 'p50':...
preProcessingMillis                                                      0
computeMillis                                                            4
postProcessingMillis                                                     1
writeMillis                                                             30
nodePropertiesWritten                                                  356
configuration            {'writeConcurrency': 4, 'seedProperty': None, ...
Name: 0, dtype: object

In [19]:
result = gds.run_cypher("""
MATCH (e:EntityMention)
WITH e.wcc AS community, count(*) AS size
RETURN community, size
ORDER BY size DESC
LIMIT 20
""")

print(result)

    community  size
0          15    57
1           1    36
2           2    22
3          16    19
4          18    15
5          71     9
6           5     7
7         218     6
8         176     5
9          39     5
10         89     5
11        157     4
12         84     4
13        181     3
14        210     3
15         27     3
16        118     3
17         45     3
18         81     3
19         36     3


In [20]:
SYSTEM_PROMPT = """
You are an expert in medical entity resolution.

Your task is to identify entity mentions that refer to the same real-world entity.

Each entity has:
- mention_id
- name
- type
- description

Rules:

1. A mention_id can belong to ONE AND ONLY ONE MergeGroup.

2. Respone members is list mention_id that refer to the same real-world entity.

3. Merge entities with minor spelling differences.

4. Merge abbreviations and full names.

5. Merge different languages referring to the same entity.

6. Use description to understand meaning.

Input entities:

{input_text}
"""

In [21]:
from typing import List
from pydantic import BaseModel, Field


class MergeGroup(BaseModel):
    canonical_name: str = Field(
        description="Canonical entity name."
    )

    canonical_type: str = Field(
        description="Canonical entity type."
    )

    canonical_description: str = Field(
        description="Merged description of the entity."
    )

    members: List[str] = Field(
        description="List of mention_id"
    )


class EntityResolutionResult(BaseModel):
    groups: List[MergeGroup]

In [22]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key="lm-studio",
    base_url="http://localhost:8000/v1",
    model="qwen-3-1.7b",
    temperature=0,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    },
)

structured_llm = llm.with_structured_output(EntityResolutionResult)

In [23]:
def build_prompt(text: str) -> str:
    return SYSTEM_PROMPT.format(
        input_text=text
    )

In [24]:
def resolve_one_community(community_id, entities):

    try:
        prompt = build_prompt(entities)
        response = structured_llm.invoke([
            {
                "role": "system",
                "content": "You are an expert in information extraction."
            },
            {
                "role": "user",
                "content": prompt
            }
        ])
        return {
            "community": community_id,
            "groups": response.groups
        }

    except Exception as e:

        print(f"Community {community_id} failed:", e)

        return None

In [25]:
communities = gds.run_cypher("""

MATCH (e:EntityMention)

WITH e.wcc AS community,

collect({

    mention_id:e.mention_id,

    name:e.name,

    type:e.type,

    description:e.description

}) AS entities

WHERE size(entities) > 1

RETURN community, entities

""")

In [26]:
from pprint import pprint

for _, row in communities.iterrows():
    print("=" * 80)
    print("Community:", row.community)
    pprint(row.entities)

Community: 1
[{'description': "A hospital where the patient's question was referred for "
                 'further assessment.',
  'mention_id': '12_1',
  'name': 'Bệnh viện Đa khoa Tâm Anh',
  'type': 'hospital'},
 {'description': "A hospital providing medical care for women's health issues, "
                 'particularly related to pregnancy and childbirth.',
  'mention_id': '11_0',
  'name': 'Bệnh viện Đa khoa Tâm Anh',
  'type': 'hospital'},
 {'description': 'A hospital that provides medical care, specifically '
                 "addressing the patient's concerns about radiation exposure "
                 'during pregnancy.',
  'mention_id': '2_0',
  'name': 'Bệnh viện Đa khoa Tâm Anh',
  'type': 'hospital'},
 {'description': 'A hospital that provides medical care and services, '
                 'including prenatal care and consultation for pregnant women.',
  'mention_id': '0_0',
  'name': 'Bệnh viện Đa khoa Tâm Anh',
  'type': 'hospital'},
 {'description': "A hospital provid

In [27]:
from concurrent.futures import ThreadPoolExecutor, as_completed

MAX_WORKERS = 16

results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = {

        executor.submit(
            resolve_one_community,
            row.community,
            row.entities
        ): row.community

        for _, row in communities.iterrows()

    }

    total = len(futures)

    for idx, future in enumerate(as_completed(futures), 1):

        result = future.result()

        if result:
            results.append(result)

        print(f"{idx}/{total}")


# results = []
# for _, row in communities.iterrows():
#     print(f"Processing community {row.community}...")
#     result = resolve_one_community(row.community, row.entities)
#     if result:
#         results.append(result)


1/36
2/36
3/36
4/36
5/36
6/36
7/36
8/36
9/36
10/36
11/36
12/36
13/36
14/36
15/36
16/36
17/36
18/36
19/36
20/36
21/36
22/36
23/36
24/36
25/36
26/36
27/36
28/36
29/36
30/36
31/36
32/36
33/36
34/36
35/36
Community 15 failed: Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=1050, prompt_tokens=3046, total_tokens=4096, completion_tokens_details=None, prompt_tokens_details=None)
36/36


In [28]:
merge_data = []
used_mentions = set()

for community in results:
    for group in community["groups"]:

        # Chỉ giữ những mention chưa từng được dùng
        members = [
            m for m in group.members
            if m not in used_mentions
        ]

        # MergeGroup phải có ít nhất 2 entity
        if len(members) < 2:
            continue

        merge_data.append({
            "target": members[0],
            "members": members,
            "canonical_name": group.canonical_name,
            "canonical_type": group.canonical_type,
            "canonical_description": group.canonical_description,
        })

        # Đánh dấu đã sử dụng
        used_mentions.update(members)

In [29]:
all_members = []

for g in merge_data:
    all_members.extend(g["members"])

from collections import Counter

dup = {
    k: v
    for k, v in Counter(all_members).items()
    if v > 1
}

print(dup)

{}


In [30]:
from pprint import pprint

pprint(merge_data)

[{'canonical_description': 'The unborn child, from the time of conception '
                           'until birth. It is the subject of prenatal care '
                           'and monitoring.',
  'canonical_name': 'Thai nhi',
  'canonical_type': 'body_part',
  'members': ['1_6', '31_2', '30_2', '28_1', '29_1'],
  'target': '1_6'},
 {'canonical_description': 'A female individual, likely the patient, who is '
                           'undergoing hormonal testing and is experiencing '
                           'polycystic ovary syndrome (PCOS) and is seeking '
                           'reproductive healthcare.',
  'canonical_name': 'Chị',
  'canonical_type': 'person',
  'members': ['5_1', '17_0', '20_0', '26_0', '22_6', '19_0', '40_0', '10_0'],
  'target': '5_1'},
 {'canonical_description': 'The spouse of the patient, involved in the '
                           'reproductive health assessment and evaluation.',
  'canonical_name': 'Vợ',
  'canonical_type': 'person',
  'members'

In [31]:
cypher="""UNWIND $groups AS group

MATCH (e:EntityMention)
WHERE e.mention_id IN group.members

WITH group, collect(e) AS nodes

CALL apoc.refactor.mergeNodes(
    nodes,
    {
        properties:{
            `.*`:'discard'
        }
    }
)
YIELD node

SET
    node.name = group.canonical_name,
    node.type = group.canonical_type,
    node.description = group.canonical_description

RETURN count(node)"""

In [32]:
gds.run_cypher(
    cypher,
    params={"groups": merge_data}
)

,count(node)
0,24


In [33]:
cypher="""MATCH (a:EntityMention)-[r]->(b:EntityMention)

WITH
    a,
    b,
    type(r) AS rel_type,
    r.relation AS relation,
    collect(r) AS rels,
    sum(r.strength) AS total_strength

WHERE size(rels) > 1

CALL apoc.refactor.mergeRelationships(
    rels,
    {
        properties:"discard"
    }
)
YIELD rel

SET rel.strength = total_strength

RETURN count(*)
"""

In [34]:
gds.run_cypher(
    cypher,
    params={"groups": merge_data}
)

,count(*)
0,20


In [35]:
gds.close()

Constructing and summarizing communities

In [1]:
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "123456789"


In [2]:
from graphdatascience import GraphDataScience

gds = GraphDataScience(
    URI,
    auth=(USERNAME, PASSWORD)
)

d:\HocTap\ChatBot\Medical_Chatbot\Data_LakeHouse\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
if gds.graph.exists("communities")["exists"]:
    gds.graph.drop("communities")

G, result = gds.graph.project(
    "communities",
    "EntityMention",
    {
        "_ALL_": {
            "type": "*",
            "orientation": "UNDIRECTED",
            "properties": "weight",
        }
    },
)

print(result)

nodeProjection            {'EntityMention': {'properties': {}, 'label': ...
relationshipProjection    {'_ALL_': {'orientation': 'UNDIRECTED', 'aggre...
graphName                                                       communities
nodeCount                                                               251
relationshipCount                                                       522
projectMillis                                                            25
Name: 0, dtype: object


In [4]:
wcc = gds.wcc.stats(G)

print(f"Component count: {wcc['componentCount']}")
print(f"Component distribution: {wcc['componentDistribution']}")

Component count: 33
Component distribution: {'p1': 1, 'max': 176, 'p5': 1, 'p90': 6, 'p50': 1, 'p95': 7, 'p10': 1, 'p75': 3, 'p99': 176, 'p25': 1, 'min': 1, 'mean': 7.606060606060606, 'p999': 176}


In [5]:
leiden_result = gds.leiden.write(
    G,
    writeProperty="communities",
    includeIntermediateCommunities=True,
    relationshipWeightProperty="weight"
)

print(leiden_result)

ranLevels                                                                3
didConverge                                                           True
nodeCount                                                              251
communityCount                                                          44
preProcessingMillis                                                      0
computeMillis                                                          163
postProcessingMillis                                                    14
writeMillis                                                             27
nodePropertiesWritten                                                  251
communityDistribution    {'p1': 1, 'max': 55, 'p5': 1, 'p90': 12, 'p50'...
modularities             [0.6694275662745627, 0.7173008520088625, 0.719...
modularity                                                        0.719674
configuration            {'writeConcurrency': 4, 'consecutiveIds': Fals...
Name: 0, dtype: object


In [6]:
cypher = """
MATCH (e:EntityMention)

UNWIND range(0, size(e.communities)-1) AS level

MERGE (c:Community {
    id: toString(level) + "-" + toString(e.communities[level])
})
ON CREATE SET
    c.level = level,
    c.community_id = e.communities[level]

MERGE (e)-[:IN_COMMUNITY]->(c)
"""

gds.run_cypher(cypher)

""


In [7]:
cypher = """
MATCH (e:EntityMention)

UNWIND range(1, size(e.communities)-1) AS level

MERGE (parent:Community {
    id: toString(level-1) + "-" + toString(e.communities[level-1])
})

MERGE (child:Community {
    id: toString(level) + "-" + toString(e.communities[level])
})

MERGE (parent)-[:IN_COMMUNITY]->(child)
"""

gds.run_cypher(cypher)

""


In [8]:
cypher = """
MATCH (c:Community)
<-[:IN_COMMUNITY]-
(:EntityMention)
<-[:MENTIONS]-
(:Chunk)
<-[:HAS_CHUNK]-
(d:Document)

WITH c, count(DISTINCT d) AS rank

SET c.community_rank = rank
"""

gds.run_cypher(cypher)


""


In [9]:
import pandas as pd
import numpy as np


community_size = gds.run_cypher(
    """
MATCH (c:Community)<-[:IN_COMMUNITY*]-(e:EntityMention)
WITH c, count(distinct e) AS entities
RETURN split(c.id, '-')[0] AS level, entities
"""
)
community_size_df = pd.DataFrame.from_records(community_size)
percentiles_data = []
for level in community_size_df["level"].unique():
    subset = community_size_df[community_size_df["level"] == level]["entities"]
    num_communities = len(subset)
    percentiles = np.percentile(subset, [25, 50, 75, 90, 99])
    percentiles_data.append(
        [
            level,
            num_communities,
            percentiles[0],
            percentiles[1],
            percentiles[2],
            percentiles[3],
            percentiles[4],
            max(subset)
        ]
    )

# Create a DataFrame with the percentiles
percentiles_df = pd.DataFrame(
    percentiles_data,
    columns=[
        "Level",
        "Number of communities",
        "25th Percentile",
        "50th Percentile",
        "75th Percentile",
        "90th Percentile",
        "99th Percentile",
        "Max"
    ],
)
percentiles_df

C:\Users\Windows\AppData\Local\Temp\ipykernel_27580\1477743568.py:12: FutureWarning: Passing a DataFrame to DataFrame.from_records is deprecated. Use set_index and/or drop to modify the DataFrame instead.
  community_size_df = pd.DataFrame.from_records(community_size)


,Level,Number of communities,25th Percentile,50th Percentile,75th Percentile,90th Percentile,99th Percentile,Max
0,2,44,1.0,3.0,5.25,15.5,51.56,55
1,0,63,1.0,3.0,4.00,6.8,26.16,46
2,1,44,1.0,3.0,5.25,15.5,41.67,55


Comunity Report

In [10]:
cypher = """
MATCH (c:Community)<-[:IN_COMMUNITY]-(e:EntityMention)
WHERE c.level IN [0,1,2]

WITH c, collect(e) AS nodes
WHERE size(nodes) > 1

CALL apoc.path.subgraphAll(
    nodes[0],
    {
        whitelistNodes: nodes
    }
)
YIELD relationships

RETURN
    c.id AS communityId,
    c.level AS level,

    [n IN nodes |
        {
            id: elementId(n),
            name: n.name,
            description: n.description
        }
    ] AS nodes,

    [r IN relationships |
        {
            start: startNode(r).name,
            type: type(r),
            end: endNode(r).name,
            description: r.description,
            weight: r.weight
        }
    ] AS rels

ORDER BY level, communityId
"""

community_info = gds.run_cypher(cypher)

community_info = community_info.to_dict("records")

print(len(community_info))
for community in community_info[:5]:
    print("=" * 80)
    print(f"Community ID: {community['communityId']}, Level: {community['level']}")
    print("Nodes:")
    for node in community["nodes"]:
        print(f"  - ID: {node['id']}, Name: {node['name']}, Description: {node['description']}")
    print("Relationships:")
    for rel in community["rels"]:
        print(f"  - Start: {rel['start']}, Type: {rel['type']}, End: {rel['end']}, Description: {rel['description']}, Weight: {rel['weight']}")

100
Community ID: 0-1, Level: 0
Nodes:
  - ID: 4:7db85d1f-e5ef-4e9b-8116-a375dae1eab3:422, Name: Cảm cúm, Description: A common respiratory infection characterized by symptoms such as fever, cough, and sore throat.
  - ID: 4:7db85d1f-e5ef-4e9b-8116-a375dae1eab3:318, Name: Hội chứng antiphospholipid, Description: A blood clotting disorder that can cause recurrent miscarriages and placental issues.
  - ID: 4:7db85d1f-e5ef-4e9b-8116-a375dae1eab3:561, Name: Trung tâm Sản phụ khoa, Description: A specialized department within the hospital that focuses on gynecological care, including examinations and health services for women.
  - ID: 4:7db85d1f-e5ef-4e9b-8116-a375dae1eab3:321, Name: Phụ nữ bị sảy thai nhiều lần, Description: A condition referring to women who have experienced multiple miscarriages without identifying a cause.
  - ID: 4:7db85d1f-e5ef-4e9b-8116-a375dae1eab3:319, Name: Bất đồng nhóm máu mẹ con, Description: A blood group incompatibility between the mother and fetus that can l

In [11]:
from pydantic import BaseModel, Field


class CommunityReport(BaseModel):

    title: str = Field(description="Short title")

    summary: str = Field(description="Community summary")

    key_entities: list[str] = Field(
        description="Important entities"
    )

    key_relationships: list[str] = Field(
        description="Important relationships"
    )

    findings: list[str] = Field(
        description="Important findings"
    )

In [12]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    api_key="lm-studio",
    base_url="http://localhost:8000/v1",
    model="qwen-3-1.7b",
    temperature=0,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    },
)

structured_llm = llm.with_structured_output(CommunityReport)


In [13]:
from langchain_core.prompts import ChatPromptTemplate

community_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
You are an expert knowledge graph analyst.

You analyze graph communities.

Rules:

- Use ONLY the provided information.
- Do NOT hallucinate.
- Explain the main topic.
- Identify important entities.
- Explain important relationships.
- Infer useful findings.

Return a structured report.
"""
        ),

        (
            "human",
            """
Community Level:
{level}

Number of entities:
{num_entities}

Number of relationships:
{num_relationships}

Entities

{entities}

Relationships

{relationships}
"""
        ),
    ]
)

In [14]:
def prepare_string(data):

    entities = []

    for node in data["nodes"]:

        text = f"- {node['name']}"

        if node.get("description"):
            text += f": {node['description']}"

        entities.append(text)

    relationships = []

    for rel in data["rels"]:

        text = (
            f"- {rel['start']} "
            f"--[{rel['type']}]--> "
            f"{rel['end']}"
        )

        if rel.get("strength") is not None:
            text += f" (strength={rel['strength']})"

        if rel.get("description"):
            text += f": {rel['description']}"

        relationships.append(text)

    return {
        "entities": "\n".join(entities),
        "relationships": "\n".join(relationships),
    }

In [15]:
def process_community(community):

    info = prepare_string(community)

    prompt = community_prompt.format_messages(
        level=community["level"],
        num_entities=len(community["nodes"]),
        num_relationships=len(community["rels"]),
        entities=info["entities"],
        relationships=info["relationships"],
    )

    response = structured_llm.invoke(prompt)

    return {
        "community": community["communityId"],
        "title": response.title,
        "summary": response.summary,
        "key_entities": response.key_entities,
        "key_relationships": response.key_relationships,
        "findings": response.findings,
    }

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

summaries = []

with ThreadPoolExecutor(max_workers=4) as executor:

    futures = [
        executor.submit(process_community, c)
        for c in community_info
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures),
        desc="Generating community reports"
    ):

        try:

            summaries.append(future.result())

        except Exception as e:

            print(e)

Generating community reports:   0%|          | 0/100 [00:00<?, ?it/s]

Generating community reports:   6%|▌         | 6/100 [00:23<04:55,  3.15s/it]

Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=726, prompt_tokens=3370, total_tokens=4096, completion_tokens_details=None, prompt_tokens_details=None)


Generating community reports:  66%|██████▌   | 66/100 [02:53<01:41,  2.98s/it]

Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=98, prompt_tokens=3998, total_tokens=4096, completion_tokens_details=None, prompt_tokens_details=None)


Generating community reports:  82%|████████▏ | 82/100 [03:59<03:23, 11.32s/it]

Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=1473, prompt_tokens=2623, total_tokens=4096, completion_tokens_details=None, prompt_tokens_details=None)


Generating community reports:  83%|████████▎ | 83/100 [04:03<02:34,  9.09s/it]

Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=98, prompt_tokens=3998, total_tokens=4096, completion_tokens_details=None, prompt_tokens_details=None)


Generating community reports: 100%|██████████| 100/100 [04:51<00:00,  2.92s/it]


In [17]:
for summary in summaries[:5]:
    print("=" * 80)
    print(f"Community ID: {summary['community']}")
    print(f"Title: {summary['title']}")
    print(f"Summary: {summary['summary']}")
    print(f"Key Entities: {summary['key_entities']}")
    print(f"Key Relationships: {summary['key_relationships']}")
    print(f"Findings: {summary['findings']}")



Community ID: 0-10
Title: Community Analysis
Summary: The community consists of two entities: 'Thuốc tẩy giun' (Drug used to treat parasitic infections) and 'Phụ nữ có thai' (Pregnant women). There is one relationship: 'RELATED_TO' between the drug and pregnant women, indicating that the drug is used by pregnant women, who are at risk of complications if they take it.
Key Entities: ['Thuốc tẩy giun', 'Phụ nữ có thai']
Key Relationships: ['Thuốc tẩy giun --[RELATED_TO]--> Phụ nữ có thai']
Findings: ["The drug 'Thuốc tẩy giun' is used by pregnant women, who are at risk of complications if they take it.", 'This relationship highlights a potential risk for pregnant women when using the drug, suggesting a need for caution or alternative treatment options.']
Community ID: 0-102
Title: Community Analysis of Reproductive Medicine
Summary: This community focuses on reproductive medicine, particularly on the role of AMH, IVF, and ovarian reserve in fertility treatments.
Key Entities: ['AMH', 'Fe

In [ ]:
def build_community_text(row):

    title = row["title"] or ""
    summary = row["summary"] or ""

    return f"""
        Title:
        {title}

        Summary:
        {summary}
        """.strip()

In [ ]:
texts = [
    build_community_text(r)
    for r in summaries
]
model = SentenceTransformer(
    "bkai-foundation-models/vietnamese-bi-encoder"
)

embeddings = model.encode(
    texts,
    batch_size=128,
    normalize_embeddings=True,
    show_progress_bar=True
).tolist()

for report, emb in zip(summaries, embeddings):
    report["embedding"] = emb

In [ ]:
save_query = """
UNWIND $rows AS row

MATCH (c:Community {id: row.community})

SET
    c.title = row.title,
    c.summary = row.summary,
    c.key_entities = row.key_entities,
    c.key_relationships = row.key_relationships,
    c.findings = row.findings,
    c.embedding = row.embedding
"""

gds.run_cypher(
    save_query,
    params={
        "rows": summaries
    }
)

""
